# From Java to Native Tensors via MLIR TOSA

## About Me

- Software developer
- Linux Kernel driver development in [FRITZ!Box](https://fritz.com/) (2009-2014)
- Java VM Enginner at [MicroDoc](https://microdoc.com) (2015-now)
    - Operating System / CPU ports
    - GraalVM Native Image AOT Compilation

## LLVM Meetup — Project Babylon Demo

In this talk, we’ll explore a toy project that translates Java code into MLIR for the TOSA dialect, which is used for AI and accelerator workloads. We’ll briefly introduce current approaches (e.g., TornadoVM and Babylon/HAT), before looking at how this project reuses existing compiler infrastructure instead of building new backends. Finally, we’ll touch on whether DSLs should be embedded in general-purpose languages like Java or designed as standalone languages.


---

> **Environment required**
> ```
> export JAVA_HOME=/path/to/babylon/jdk
> ```
> Build the jar: `mvn package -Pwith-examples -DskipTests`

In [8]:
%jars ../cr-examples/mlir-tosa/target/mlir-tosa-1.0-SNAPSHOT.jar

---
## ☕ #0 — Context: Java in 2026


<table style="border-collapse: collapse;">
  <tr>
    <td style="border: 0;">
      <img src="images/birthday.png" title="Java birthday!" alt="Java turns">
    </td>
    <td style="border: 0;">
      <strong><b>Java turns 31!!!</b><br><br>...and it is changing faster than at any point in its history.</strong>
    </td>
  </tr>
</table>

but...

![TIOBE index 2026](images/tiobe2026.png "TIOBE index 02/04/2026")

### Paving the ramp...

- No mandatory class/main in some environments → small programs can run as direct statements
- Automatic structure generation → compiler/tools create the class and entry point behind the scenes
- Script-like execution support → single-file run and REPL-style execution remove manual compile/run steps

In [9]:
//Classic Java Hello World

public class HelloWorld {
    public static void main(String[] args) {
        System.out.println("Hello, world");
    }
}


In [10]:
//Modern Java Hello World

void main() {
    System.out.println("Hello, world");
}

main();

Hello, world


### Deliver faster!

**Release cadence**: since Java 9 (2017), releases ship every 6 months on a fixed calendar schedule
(March and September), regardless of which features are ready. Features that need more time ship as
*preview* — available but not final — and graduate across releases. This decouples language
evolution from big-bang versioning.

**The platform projects relevant to us**:

| Project | Status | What it does |
|---|---|---|
| **Panama** | GA — Java 22 | Foreign Function & Memory API: native interop without JNI |
| **Loom** | GA — Java 21 | Virtual threads: millions of lightweight threads |
| **Valhalla** | In progress | Value types: flat, dense memory layout for objects |
| **Babylon** | Incubating — Java 26 | Code reflection: inspect method bodies as a compiler IR |

Panama gives us the **zero-copy calling convention** used in this demo.
Babylon gives us the **code model** we compile to TOSA.
Valhalla will eventually give tensors a better memory story.

Java is not the Java of 2005. The platform is being modernised specifically to close the gap
with systems languages — and to plug into ecosystems like LLVM.

### Java and AI 2026

AI Orchestration over Training: Java's core strength lies in its maturity, reliability, and security. It is predominantly used as the "chassis" for connecting Language Learning Models (LLMs) to complex business logic, retrieval-augmented generation (RAG), and agentic workflows.

But the hot path — tensor computation — usually crosses a boundary into Python, C++, CUDA, vendor SDKs, or a native inference runtime.

### Existing Java approaches

Java has had several attempts at this problem: keep the programming model productive, but move the hot numerical code closer to the hardware.

| Approach | Basic idea | Technology | Trade-off |
|---|---|---|---|
| **TornadoVM** | Run selected Java code on heterogeneous hardware by analysing bytecode and JIT-compiling kernels for devices such as GPUs. | TornadoVM sits on top of the JVM using JVMCI (JVM Compiler Interface), which is also used by GraalVM. | Powerful, but it needs a specialised runtime, task graph, and backend stack. |
| **Babylon/HAT** | Use Babylon code reflection to capture Java method bodies as structured code, then lower accelerator-oriented kernels from that representation. | Project babylon | Much closer to the Java language model, but still focused on building an accelerator programming stack. |

**Tornado VM**

The key idea in TornadoVM is that developers write standard Java code and mark data-parallel loops with annotations like @Parallel, allowing the TornadoVM compiler to automatically transform those regions into GPU or multi-core kernels. The runtime then analyzes these parallel loops, maps iteration spaces onto hardware threads, manages data movement between CPU and accelerators, and generates optimized OpenCL or PTX code for execution, enabling heterogeneous computing without requiring developers to write CUDA or OpenCL directly.

```java
private static void mxmLoop(Matrix2DFloat A, Matrix2DFloat B, Matrix2DFloat C, final int size) {
        for (@Parallel int i = 0; i < size; i++) {
            for (@Parallel int j = 0; j < size; j++) {
                float sum = 0.0f;
                for (int k = 0; k < size; k++) {
                    sum += A.get(i, k) * B.get(k, j);
                }
                C.set(i, j, sum);
            }
        }
    }
```

**Babylon/HAT**

The key idea is that developers can express parallel computations using Java methods annotated for “code reflection,” and the HAT runtime maps thread indices, memory layouts, and kernel launches onto GPU execution.
```java
@CodeReflection
static void vectorMulKernel(KernelContext kc,
                            F32Array a,
                            F32Array b,
                            F32Array c) {

    int i = kc.x;

    if (i < a.length()) {
        c.set(i, a.get(i) * b.get(i));
    }
}
```



Both approaches are important because they show that Java can be more than a host language that calls into native libraries. Java code itself can become compiler input.

![Standards](images/skcd_standards.png)

This demo takes a narrower path. It does **not** try to build a full Java accelerator runtime. Instead, it asks a smaller question:

> If Babylon can give us a structured Java code model, can we translate a tensor-shaped subset of Java directly into an existing MLIR dialect?


---
## 🌍 #0 — More context: DSLs for Tensor Computation

**The challenge**: write high-level tensor operations in a productive language, get native hardware performance.

This is not a new problem. The Python ecosystem has been solving it for years.

### PyTorch `torch.compile`

```python
@torch.compile
def model(x, w):
    return torch.relu(x @ w)
```

Traces at runtime via Python bytecode inspection → FX graph → TorchInductor → Triton / C++.
The decorator is a library hook — the compiler sees no special semantics.

### OpenAI Triton

```python
@triton.jit
def add_kernel(x_ptr, y_ptr, out_ptr, n, BLOCK: tl.constexpr):
    pid = tl.program_id(0)
    offs = pid * BLOCK + tl.arange(0, BLOCK)
    tl.store(out_ptr + offs, tl.load(x_ptr + offs) + tl.load(y_ptr + offs))
```

A true DSL: `@triton.jit` captures the Python AST → Triton IR → LLVM → PTX.
Closer to a compiler — but relying on runtime AST inspection of a dynamic language.

### JAX / XLA / StableHLO

```python
@jax.jit
def model(x, w):
    return jax.nn.relu(x @ w)
```

JAX traces through the function using abstract values → HLO graph → **StableHLO** (a stable MLIR dialect) → XLA → hardware-specific code (TPU, GPU, CPU).
StableHLO is the closest prior art to what we are building here — but it lives entirely in the Python/C++ world.

### The common pattern

```
annotate function  →  capture IR  →  lower to native
```

Python can do this because it is interpreted and its AST/trace is available at runtime.

### The gap in Java

Java compiles to Java bytecode. `java.lang.reflect` gives you method *signatures* — not bodies.
There is no standard way to ask the JVM: *"what does this method actually do?"*

You can inspect bytecode with ASM, or trace execution, but there is no structured, type-safe IR
exposed as a platform feature.

**What raw bytecode looks like**

Without Babylon, `javap -c` is the closest you can get to inspecting a method body.
Below we compile a simple multiply-accumulate function on-the-fly and disassemble it:

In [11]:
import javax.tools.*;
import java.nio.file.*;

// Plain Java — no @Reflect, no Babylon
String source = """
    class MacExample {
        static int mac(int x, int y, int z) {
            return x * y + z;
        }
        static int loop(int n) {
            int acc = 0;
            for (int i = 0; i < n; i++) {
                acc = mac(acc, i, 1);
            }
            return acc;
        }
    }
    """;

var tmpDir = Files.createTempDirectory("bytecode-demo");
var srcFile = tmpDir.resolve("MacExample.java");
Files.writeString(srcFile, source);

ToolProvider.getSystemJavaCompiler().run(null, null, null, srcFile.toString());

var javap = Path.of(System.getProperty("java.home"), "bin", "javap");
var proc = new ProcessBuilder(javap.toString(), "-c", tmpDir.resolve("MacExample.class").toString())
    .redirectErrorStream(true)
    .start();
System.out.println(new String(proc.getInputStream().readAllBytes()));

Compiled from "MacExample.java"
class MacExample {
  MacExample();
    Code:
         0: aload_0
         1: invokespecial #1                  // Method java/lang/Object."<init>":()V
         4: return

  static int mac(int, int, int);
    Code:
         0: iload_0
         1: iload_1
         2: imul
         3: iload_2
         4: iadd
         5: ireturn

  static int loop(int);
    Code:
         0: iconst_0
         1: istore_1
         2: iconst_0
         3: istore_2
         4: iload_2
         5: iload_0
         6: if_icmpge     22
         9: iload_1
        10: iload_2
        11: iconst_1
        12: invokestatic  #7                  // Method mac:(III)I
        15: istore_1
        16: iinc          2, 1
        19: goto          4
        22: iload_1
        23: ireturn
}



The bytecode is a **stack machine sequence**: `iload`, `imul`, `iadd`, `ireturn`.
Types are encoded in the instruction prefix (`i` = int) but source structure — the two-step `x*y` then `+z` — is erased into a flat sequence of stack operations.
There is no structured IR you can walk, pattern-match, or rewrite.

**Until Project Babylon.**

---
## 🪞 #1 — Project Babylon: Java as a Compiler Frontend

Standard Java reflection exposes *structure* (class, method signatures) but not *behaviour*.

Project Babylon adds `@Reflect`: an opt-in annotation that tells the compiler to record enough
metadata so that the method body can be reconstructed at runtime as a structured IR — a `FuncOp`.

Think of it as Clang emitting LLVM IR from C, but from Java method bodies.

> **Why not just read the bytecode?**
> Bytecode is below source level — types are erased, generics are gone, control flow is encoded
> as jump targets. Babylon's code model sits *above* bytecode: it preserves Java types, generic
> signatures, and source-level control flow, making it suitable as a compiler IR rather than a
> disassembler output.

In [12]:
import jdk.incubator.code.Reflect;
import jdk.incubator.code.Op;
import jdk.incubator.code.dialect.core.CoreOp;
import mlir.tosa.*;

// A simple tensor addition — annotated so the compiler records its body
class AddDemo {
    @Reflect
    static Tensor<Float> add(Tensor<Float> x, Tensor<Float> y) {
        return TosaOperators.Add(x, y);
    }
}

var addMethod = AddDemo.class.getDeclaredMethod("add", Tensor.class, Tensor.class);

// From standard java.lang.reflect.Method -> Babylon CoreOp.FuncOp
CoreOp.FuncOp funcOp = Op.ofMethod(addMethod).orElseThrow();
System.out.println(funcOp.toText());

func @loc="23:5:string:///REPL/$JShell$47.java" @"add" (%0 : java.type:"mlir.tosa.Tensor<java.lang.Float>", %1 : java.type:"mlir.tosa.Tensor<java.lang.Float>")java.type:"mlir.tosa.Tensor<java.lang.Float>" -> {
    %2 : Var<java.type:"mlir.tosa.Tensor<java.lang.Float>"> = var %0 @loc="23:5" @"x";
    %3 : Var<java.type:"mlir.tosa.Tensor<java.lang.Float>"> = var %1 @loc="23:5" @"y";
    %4 : java.type:"mlir.tosa.Tensor<java.lang.Float>" = var.load %2 @loc="25:34";
    %5 : java.type:"mlir.tosa.Tensor<java.lang.Float>" = var.load %3 @loc="25:37";
    %6 : java.type:"mlir.tosa.Tensor<java.lang.Float>" = invoke %4 %5 @loc="25:16" @java.ref:"mlir.tosa.TosaOperators::Add(mlir.tosa.Tensor, mlir.tosa.Tensor):mlir.tosa.Tensor";
    return %6 @loc="25:9";
};


The IR above is in source-like form. Applying an SSA transform makes data dependencies explicit —
each value is defined exactly once, control flow is explicit via block parameters.

In [13]:
import jdk.incubator.code.dialect.core.SSA;

CoreOp.FuncOp ssaOp = SSA.transform(funcOp);
System.out.println(ssaOp.toText());

func @loc="23:5:string:///REPL/$JShell$47.java" @"add" (%0 : java.type:"mlir.tosa.Tensor<java.lang.Float>", %1 : java.type:"mlir.tosa.Tensor<java.lang.Float>")java.type:"mlir.tosa.Tensor<java.lang.Float>" -> {
    %2 : java.type:"mlir.tosa.Tensor<java.lang.Float>" = invoke %0 %1 @loc="25:16" @java.ref:"mlir.tosa.TosaOperators::Add(mlir.tosa.Tensor, mlir.tosa.Tensor):mlir.tosa.Tensor";
    return %2 @loc="25:9";
};


## ⚙️ #2 — Writing a TOSA Backend

TOSA (Tensor Operator Set Architecture) is an MLIR dialect that defines a standardized, hardware-independent set of tensor operations for machine learning. It serves as an intermediate representation between ML frameworks and hardware backends, enabling portable optimization and code generation across CPUs, GPUs, and AI accelerators.

[https://www.mlplatform.org/tosa/tosa_spec.html](https://www.mlplatform.org/tosa/tosa_spec.html)

### Why TOSA?

| | ONNX | TOSA |
|---|---|---|
| Ops | 180+ | ~80 |
| Semantics | Framework-dependent | Fully specified |
| Integration | External format | First-class MLIR dialect |
| Lowering | Varies | Direct path to Linalg → LLVM |

`TosaCodeGenerator` walks the `CoreOp.FuncOp` tree and emits TOSA ops via the native MLIR C API.

### Simple example: element-wise Add

**Stage 0 — TOSA**: The Java `TosaOperators.Add()` call maps to a single `tosa.add` op. Shapes stay dynamic (`?xf32`) — TOSA does not require size information at this level. Broadcasting, indexing, and memory allocation are all implicit; the TOSA spec defines their semantics precisely, but the representation stays intentionally high-level.

In [14]:
// Walk the code model -> emit TOSA MLIR
String addTosa = TosaCodeGenerator.generateTosa(addMethod);
System.out.println(addTosa);

module {
  func.func @add(%arg0: tensor<?xf32>, %arg1: tensor<?xf32>) -> tensor<?xf32> {
    %0 = tosa.add %arg0, %arg1 : (tensor<?xf32>, tensor<?xf32>) -> tensor<?xf32>
    return %0 : tensor<?xf32>
  }
}



### More complex example: Conv2D + ReLU

The same code generator handles higher-level ops — Conv2D lowers to a single `tosa.conv2d`.

In [15]:
class ConvDemo {
    @Reflect
    static Tensor<Float> conv(Tensor<Float> input, Tensor<Float> weight, Tensor<Float> bias) {
        return TosaOperators.Conv2D(
            input, weight, bias,
            new long[]{0, 0, 0, 0},  // padding
            new long[]{1, 1},         // stride
            new long[]{1, 1}          // dilation
        ).relu();
    }
}

var convMethod = ConvDemo.class.getDeclaredMethod("conv", Tensor.class, Tensor.class, Tensor.class);
var convFuncOp = Op.ofMethod(convMethod).orElseThrow();

// tosa.conv2d requires static shapes — dynamic shapes fail MLIR verification
long[][] shapes = {
    {1, 6, 6, 1},  // input  [N, H, W, C]
    {2, 3, 3, 1},  // weight [OC, KH, KW, IC]  (TOSA OHWI layout)
    {2}            // bias   [OC]
};
System.out.println(TosaCodeGenerator.generateTosa(convFuncOp, "conv", shapes));

module {
  func.func @conv(%arg0: tensor<1x6x6x1xf32>, %arg1: tensor<2x3x3x1xf32>, %arg2: tensor<2xf32>) -> tensor<?x?x?x?xf32> {
    %0 = tosa.conv2d %arg0, %arg1, %arg2 {dilation = array<i64: 1, 1>, pad = array<i64: 0, 0, 0, 0>, stride = array<i64: 1, 1>} : (tensor<1x6x6x1xf32>, tensor<2x3x3x1xf32>, tensor<2xf32>) -> tensor<?x?x?x?xf32>
    %1 = tosa.clamp %0 {max_fp = 3.40282347E+38 : f32, max_int = 9223372036854775807 : i64, min_fp = 0.000000e+00 : f32, min_int = 0 : i64} : (tensor<?x?x?x?xf32>) -> tensor<?x?x?x?xf32>
    return %1 : tensor<?x?x?x?xf32>
  }
}



---
## ⬇️ #3 — The Lowering Pipeline

The entire lowering pipeline runs on standard **LLVM/MLIR tools available in Debian** — no custom compiler build required. On Debian Trixie (and derivatives), `apt install mlir-19-tools` provides `mlir-opt`, `mlir-translate`, and `mlir-cpu-runner` at LLVM 19.1.7.

[LLVM Lighthouse](https://github.com/llvm/lighthouse) is a proposed MLIR "guiding project" in the LLVM ecosystem. It is a shared environment for integrating, testing, and demonstrating MLIR-based compiler pipelines. I got some ideas from [https://github.com/llvm/lighthouse/blob/main/examples/cpu/x86/matmul.py](https://github.com/llvm/lighthouse/blob/main/examples/cpu/x86/matmul.py)

All driven by `mlir-opt` + `mlir-translate` + `clang` invoked from Java via `ProcessBuilder`:

### Stage 1: TOSA → Linalg

`generateAtStage(..., Stage.LINALG)` runs only the TOSA-to-Linalg portion — stopping before bufferization and without vectorization:

```bash
mlir-opt add.mlir \
  --pass-pipeline="builtin.module(
    func.func(tosa-to-linalg-named),
    func.func(tosa-to-linalg),
    func.func(tosa-to-arith),
    func.func(tosa-to-tensor),
    func.func(linalg-fuse-elementwise-ops),
    func.func(linalg-generalize-named-ops)
  )" \
  -o add_linalg.mlir
```

**What changed**: `tosa.add` has been replaced by concrete `linalg.generic` ops with explicit indexing maps:

- **Affine maps**: `#map` (constant index `0`, for broadcasting) and `#map1` (identity) encode how each operand's indices are computed for every output element.
- **Broadcasting made explicit**: two `scf.if` blocks check whether each input has `size == 1`; if so, a broadcast `linalg.generic` expands it to the output shape before the add.
- **The addition**: a `linalg.generic` with `arith.addf` in its body, with identity indexing maps on all three operands (two inputs, one output).

Still in **tensor semantics** — the result tensor is described structurally, with no heap allocation visible. `one-shot-bufferize` (next step) decides what to allocate in-place vs. what needs a fresh buffer.

In [16]:
var compiler = new TosaCompiler();

// Run the tosa-to-linalg* passes only; stop before bufferization
System.out.println(compiler.generateAtStage(addMethod, TosaCompiler.Stage.LINALG));

#map = affine_map<(d0) -> (0)>
#map1 = affine_map<(d0) -> (d0)>
module {
  func.func @add(%arg0: tensor<?xf32>, %arg1: tensor<?xf32>) -> tensor<?xf32> {
    %c1 = arith.constant 1 : index
    %c0 = arith.constant 0 : index
    %dim = tensor.dim %arg0, %c0 : tensor<?xf32>
    %dim_0 = tensor.dim %arg1, %c0 : tensor<?xf32>
    %0 = arith.maxui %dim, %dim_0 : index
    %dim_1 = tensor.dim %arg0, %c0 : tensor<?xf32>
    %1 = arith.cmpi eq, %dim_1, %c1 : index
    %2 = scf.if %1 -> (tensor<?xf32>) {
      %7 = tensor.empty(%0) : tensor<?xf32>
      %8 = linalg.generic {indexing_maps = [#map, #map1], iterator_types = ["parallel"]} ins(%arg0 : tensor<?xf32>) outs(%7 : tensor<?xf32>) {
      ^bb0(%in: f32, %out: f32):
        linalg.yield %in : f32
      } -> tensor<?xf32>
      scf.yield %8 : tensor<?xf32>
    } else {
      scf.yield %arg0 : tensor<?xf32>
    }
    %dim_2 = tensor.dim %arg1, %c0 : tensor<?xf32>
    %3 = arith.cmpi eq, %dim_2, %c1 : index
    %4 = scf.if %3 -> (tensor<?xf32>)

### Stage 2: Linalg → LLVM Dialect

**What changed**: after bufferization, loop generation, and all conversions — ready for `mlir-translate`.

- **Tensors → memrefs**: `tensor<?xf32>` is now `memref<?xf32>`, a concrete memory descriptor carrying `{allocated_ptr, aligned_ptr, offset, size[0], stride[0]}`.
- **Structured loops → flat branches**: `scf.for` / `scf.if` became `llvm.br` / `llvm.cond_br`; all control flow is now an unstructured CFG.
- **Primitive arithmetic**: the scalar addition is now `llvm.fadd`; loads and stores are explicit `llvm.load` / `llvm.store` with GEP-computed addresses.

This is legal LLVM IR expressed in MLIR operation form — `mlir-translate --mlir-to-llvmir` converts it verbatim to the `.ll` text format.

In [17]:
System.out.println(compiler.generateAtStage(addMethod, TosaCompiler.Stage.LLVM_DIALECT));

module {
  llvm.func @malloc(i64) -> !llvm.ptr
  llvm.func @add(%arg0: !llvm.ptr, %arg1: !llvm.ptr, %arg2: i64, %arg3: i64, %arg4: i64, %arg5: !llvm.ptr, %arg6: !llvm.ptr, %arg7: i64, %arg8: i64, %arg9: i64) -> !llvm.struct<(ptr, ptr, i64, array<1 x i64>, array<1 x i64>)> {
    %0 = llvm.mlir.undef : !llvm.struct<(ptr, ptr, i64, array<1 x i64>, array<1 x i64>)>
    %1 = llvm.insertvalue %arg5, %0[0] : !llvm.struct<(ptr, ptr, i64, array<1 x i64>, array<1 x i64>)> 
    %2 = llvm.insertvalue %arg6, %1[1] : !llvm.struct<(ptr, ptr, i64, array<1 x i64>, array<1 x i64>)> 
    %3 = llvm.insertvalue %arg7, %2[2] : !llvm.struct<(ptr, ptr, i64, array<1 x i64>, array<1 x i64>)> 
    %4 = llvm.insertvalue %arg8, %3[3, 0] : !llvm.struct<(ptr, ptr, i64, array<1 x i64>, array<1 x i64>)> 
    %5 = llvm.insertvalue %arg9, %4[4, 0] : !llvm.struct<(ptr, ptr, i64, array<1 x i64>, array<1 x i64>)> 
    %6 = llvm.mlir.undef : !llvm.struct<(ptr, ptr, i64, array<1 x i64>, array<1 x i64>)>
    %7 = llvm.insertv

### Stage 3: LLVM IR (`mlir-translate --mlir-to-llvmir`)

**What changed**: standard LLVM IR — the final text form before `clang`:

- **Function signature**: each `tensor<?xf32>` parameter is now the C struct `{ ptr, ptr, i64, [1 x i64], [1 x i64] }` — (allocated_ptr, aligned_ptr, offset, size[0], stride[0]). This is exactly the layout that `CompiledFunction.buildFunctionDescriptor()` describes to the FFM `Linker` on the Java side.
- **Output allocation**: `malloc` followed by 64-byte pointer alignment arithmetic materializes the result buffer.
- **The scalar loop**: `fadd float` inside an `icmp slt` / `phi`-based loop — `clang -O3 -march=native` will auto-vectorize this into AVX2/AVX-512 instructions.
- **Struct access**: `insertvalue` / `extractvalue` instructions pack and unpack the memref descriptor fields inline (no actual struct in memory — these fold into register moves).

In [18]:
System.out.println(compiler.generateAtStage(addMethod, TosaCompiler.Stage.LLVM_IR));

; ModuleID = 'LLVMDialectModule'
source_filename = "LLVMDialectModule"

declare ptr @malloc(i64)

define { ptr, ptr, i64, [1 x i64], [1 x i64] } @add(ptr %0, ptr %1, i64 %2, i64 %3, i64 %4, ptr %5, ptr %6, i64 %7, i64 %8, i64 %9) {
  %11 = insertvalue { ptr, ptr, i64, [1 x i64], [1 x i64] } undef, ptr %5, 0
  %12 = insertvalue { ptr, ptr, i64, [1 x i64], [1 x i64] } %11, ptr %6, 1
  %13 = insertvalue { ptr, ptr, i64, [1 x i64], [1 x i64] } %12, i64 %7, 2
  %14 = insertvalue { ptr, ptr, i64, [1 x i64], [1 x i64] } %13, i64 %8, 3, 0
  %15 = insertvalue { ptr, ptr, i64, [1 x i64], [1 x i64] } %14, i64 %9, 4, 0
  %16 = insertvalue { ptr, ptr, i64, [1 x i64], [1 x i64] } undef, ptr %0, 0
  %17 = insertvalue { ptr, ptr, i64, [1 x i64], [1 x i64] } %16, ptr %1, 1
  %18 = insertvalue { ptr, ptr, i64, [1 x i64], [1 x i64] } %17, i64 %2, 2
  %19 = insertvalue { ptr, ptr, i64, [1 x i64], [1 x i64] } %18, i64 %3, 3, 0
  %20 = insertvalue { ptr, ptr, i64, [1 x i64], [1 x i64] } %19, i64 %4, 4, 0
  

### memref calling convention

MLIR lowers tensor arguments to the **memref ABI**. Each tensor parameter becomes a struct:

```
{ allocated_ptr, aligned_ptr, offset, size[0]…size[n-1], stride[0]…stride[n-1] }
```

The **Foreign Function & Memory (FFM) API** in Java 22+ lets us construct and pass this struct
directly — no JNI glue, no copies. `CompiledFunction` handles the layout.

---
## ⚡ #4 — Compile and Run via FFM

`TosaCompiler.compile()` runs the full pipeline and loads the resulting `.so` into the JVM.
The returned `CompiledFunction` wraps a `MethodHandle` over the native symbol.

In [19]:
// Full pipeline: TOSA -> Linalg -> Bufferize -> SCF -> LLVM -> .so -> dlopen
var compiled = new TosaCompiler().compile(addMethod);

Tensor<Float> x = Tensor.ofFlat(1.0f, 2.0f, 3.0f);
Tensor<Float> y = Tensor.ofFlat(4.0f, 5.0f, 6.0f);

// Interpreter path (pure Java)
Tensor<Float> interpResult = AddDemo.add(x, y);
System.out.println("Interpreter: " + x + " + " + y + " = " + interpResult);

// Native path (compiled .so, invoked via FFM MethodHandle)
Tensor<Float> nativeResult = compiled.invoke(x, y);
System.out.println("Native:      " + x + " + " + y + " = " + nativeResult);

Interpreter: Tensor{shape=[3], type=FLOAT32, data=[1.0, 2.0, 3.0]} + Tensor{shape=[3], type=FLOAT32, data=[4.0, 5.0, 6.0]} = Tensor{shape=[3], type=FLOAT32, data=[5.0, 7.0, 9.0]}
Native:      Tensor{shape=[3], type=FLOAT32, data=[1.0, 2.0, 3.0]} + Tensor{shape=[3], type=FLOAT32, data=[4.0, 5.0, 6.0]} = Tensor{shape=[3], type=FLOAT32, data=[5.0, 7.0, 9.0]}


---
## 📊 #5 — MNIST Benchmark

The MNIST database is a popular machine learning dataset containing 70,000 handwritten digit images (0–9), each sized 28×28 pixels. It is widely used to train and test image recognition algorithms, especially neural networks, and is considered a standard beginner benchmark in artificial intelligence research.

![MNNIST example](images/mnist.png "MNIST example")

LeNet-style CNN on 28×28 MNIST images (NHWC format, synthetic weights):

```
Input [1, 28, 28, 1]
  → Conv2D 5×5, 6 filters  → ReLU → MaxPool 2×2  [1, 12, 12, 6]
  → Conv2D 5×5, 16 filters → ReLU → MaxPool 2×2  [1,  4,  4, 16]
  → Flatten                                        [1,  1, 256]
  → FC 256→120 + ReLU
  → FC 120→84  + ReLU
  → FC 84→10   (logits)
```

Each layer is a `@Reflect`-annotated Java method compiled to a native `.so` via the TOSA pipeline.

In [20]:
mlir.tosa.TosaMnistBenchmark.main(new String[]{});

TOSA MNIST Complete Model Benchmark: Native vs Java

Model architecture (LeNet-style CNN):
  Input:    [1, 28, 28, 1]   (MNIST image, NHWC format)
  Conv1:    5x5 kernel, 6 filters -> [1, 24, 24, 6]
  ReLU + MaxPool (2x2) -> [1, 12, 12, 6]
  Conv2:    5x5 kernel, 16 filters -> [1, 8, 8, 16]
  ReLU + MaxPool (2x2) -> [1, 4, 4, 16]
  Flatten:  [1, 1, 256]
  FC1:      256 -> 120 + ReLU
  FC2:      120 -> 84 + ReLU
  FC3:      84 -> 10 (output logits)

Configuration:
  Warmup iterations:     500
  Measurement iterations: 2000

Setting up model weights and compiling native functions...
  Total model parameters: ~44,000

Compiling native functions (this may take a moment)...
  Conv layers compiled
  FC layers compiled
Setup complete!

----------------------------------------------------------------------
Benchmark 1: Convolutional Layers Only
  [1,28,28,1] -> Conv1 -> Pool1 -> Conv2 -> Pool2 -> [1,4,4,16]
----------------------------------------------------------------------
Warming up Java 

---
## What this enables

| Step | Technology |
|---|---|
| Define operations | Java + `@Reflect` |
| Inspect IR | Project Babylon `Op.ofMethod()` |
| Emit MLIR | `TosaCodeGenerator` via native MLIR C API |
| Lower to native | `mlir-opt` pipeline + `mlir-translate` + `clang` |
| Call native code | Java FFM API (`MethodHandle`, `MemorySegment`) |

The MLIR ecosystem becomes reachable from Java without any custom compiler plugin or bytecode rewriting.

**Links**
- [Post 1 — TOSA as a compilation target](https://bruno-ah-um.github.io/posts/babylon_1_mlir_tosa/)
- [Post 2 — Code Reflection](https://bruno-ah-um.github.io/posts/babylon_2_code_reflection/)
- [Post 3 — From Java to Native](https://bruno-ah-um.github.io/posts/babylon_3_from_java_to_native/)